# Task 3 — Schema Validation

Loads `data/interim/cleaned.csv` (Task 2's output), checks every row against
a simple schema (required fields present, numeric fields actually numeric,
`tender_id` unique), and splits the result into `validated.csv` (rows that
pass every check) and `rejected.csv` (rows that fail, with the specific
reason listed) inside `data/processed/`.

Nothing is silently dropped or guessed — a rejected row keeps every original
column plus a `validation_issues` note explaining exactly why it was
rejected, so Task 4 (and anyone reviewing the data later) can see the reason.


## 1. Imports

In [7]:
from pathlib import Path

import pandas as pd


## 2. Load cleaned data

In [2]:
INTERIM_DIR = Path("../data/interim")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INTERIM_DIR / "cleaned.csv")
print(f"Loaded {len(df)} rows from cleaned.csv")
print(df.columns.tolist())


Loaded 479 rows from cleaned.csv
['tender_id', 'reference_number', 'tender_name', 'tender_number', 'multiple_search', 'agency_code', 'branch_id', 'branch_name', 'agency_name', 'tender_id_string', 'tender_status_id', 'tender_status_id_string', 'tender_status_name', 'tender_type_id', 'tender_type_name', 'technical_organization_id', 'condetional_booklet_price', 'created_at', 'last_enqueries_date', 'last_offer_presentation_date', 'offers_opening_date', 'last_enqueries_date_hijri', 'offers_opening_date_hijri', 'last_offer_presentation_date_hijri', 'inside_k_s_a', 'tender_activity_name', 'tender_activity_name_list', 'tender_activity_id', 'submition_date', 'financial_fees', 'invitation_cost', 'buying_cost', 'has_invitations', 'remaining_days', 'remaining_hours', 'remaining_mins', 'current_date', 'current_date_time', 'current_time', 'is_u_g_r_p', 'ugrp_rfx_url', 'ugrp_r_f_x_response_u_r_l', 'source_entity', 'sector']


## 3. Define the schema

- `REQUIRED_COLUMNS`: must be present and non-empty on every row.
- `NUMERIC_COLUMNS`: if present and non-null, must parse as a number.
- `tender_id` must additionally be unique across the whole file.


In [3]:
REQUIRED_COLUMNS = ["tender_id", "reference_number", "tender_name", "source_entity", "sector"]
NUMERIC_COLUMNS = ["buying_cost", "financial_fees", "invitation_cost"]


## 4. Validate every row

In [4]:
def validate_row(row):
    issues = []

    for col in REQUIRED_COLUMNS:
        if col not in row or pd.isna(row[col]) or str(row[col]).strip() == "":
            issues.append(f"missing_{col}")

    for col in NUMERIC_COLUMNS:
        if col in row and pd.notna(row[col]):
            try:
                float(row[col])
            except (ValueError, TypeError):
                issues.append(f"{col}_not_numeric")

    return issues


df["validation_issues"] = df.apply(validate_row, axis=1)


## 5. Check for duplicate `tender_id`

A duplicate `tender_id` means the same tender appears twice — this would
corrupt the upsert logic in Task 4, so it must be caught here, not later.


In [5]:
duplicate_ids = set(df["tender_id"][df["tender_id"].duplicated(keep=False)])

if duplicate_ids:
    df["validation_issues"] = df.apply(
        lambda r: r["validation_issues"] + ["duplicate_tender_id"] if r["tender_id"] in duplicate_ids else r["validation_issues"],
        axis=1,
    )

print(f"Duplicate tender_id values found: {len(duplicate_ids)}")


Duplicate tender_id values found: 0


## 6. Split into validated / rejected and save

In [6]:
df["is_valid"] = df["validation_issues"].apply(lambda x: len(x) == 0)

validated = df[df["is_valid"]].drop(columns=["validation_issues", "is_valid"])
rejected = df[~df["is_valid"]].drop(columns=["is_valid"]).copy()
rejected["validation_issues"] = rejected["validation_issues"].apply(lambda x: "; ".join(x))

validated_path = PROCESSED_DIR / "validated.csv"
rejected_path = PROCESSED_DIR / "rejected.csv"

validated.to_csv(validated_path, index=False, encoding="utf-8-sig")
rejected.to_csv(rejected_path, index=False, encoding="utf-8-sig")

print(f"Validated: {len(validated)} -> {validated_path}")
print(f"Rejected: {len(rejected)} -> {rejected_path}")

if len(rejected):
    print("\nRejection reasons breakdown:")
    print(rejected["validation_issues"].value_counts())


Validated: 479 -> ..\data\processed\validated.csv
Rejected: 0 -> ..\data\processed\rejected.csv
